# YOLO v5s - lab 8 - Task 3

# Dataset

In [2]:
# Desde kagglehub podemos descargar el dataset de sku110k con imagenes y anotaciones de las bounding boxes
# Regresa mas de la suficiente infrmacion para este proyecto

import kagglehub
path = kagglehub.dataset_download("thedatasith/sku110k-annotations")
print("Path to dataset files:", path)

100%|██████████| 13.2G/13.2G [14:24<00:00, 16.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/thedatasith/sku110k-annotations/versions/14


In [106]:
!ls {path}

convert_yolov5.ipynb  data_kaggle.yaml	README.md  SKU110K_fixed


In [107]:
import os
import random
import shutil

dataset_path = path + "/SKU110K_fixed"

img_dirs = {
    "train": os.path.join(dataset_path, "images/train"),
    "val": os.path.join(dataset_path, "images/val"),
    "test": os.path.join(dataset_path, "images/test")
}

lbl_dirs = {
    "train": os.path.join(dataset_path, "labels/train"),
    "val": os.path.join(dataset_path, "labels/val"),
    "test": os.path.join(dataset_path, "labels/test")
}

# En los requerimientos del laboratorio, se inicaban 500 de train y 100 de test y val
sample_sizes = {
    "train": 500,
    "val": 100,
    "test": 100
}

In [114]:
# Tenemos mas data de la necesaria, entonces podemos extraer unicamente los que necesitamos
# y guardarlos en una carpeta especifica

small_dataset = "/content/small_dataset2"

for split in ["train","val","test"]:
    os.makedirs(f"{small_dataset}/images/{split}", exist_ok=True)
    os.makedirs(f"{small_dataset}/labels/{split}", exist_ok=True)

for split in ["train","val","test"]:

    images = os.listdir(img_dirs[split])
    chosen = random.sample(images, sample_sizes[split])

    for img in chosen:

        img_src = os.path.join(img_dirs[split], img)
        img_dst = os.path.join(small_dataset, "images", split, img)

        label_name = os.path.splitext(img)[0] + ".txt"
        lbl_src = os.path.join(lbl_dirs[split], label_name)
        lbl_dst = os.path.join(small_dataset, "labels", split, label_name)

        shutil.copy(img_src, img_dst)

        if os.path.exists(lbl_src):
            shutil.copy(lbl_src, lbl_dst)

In [115]:
# Aqui podemos ver que se realizo correctamente
print("train images:", len(os.listdir("/content/small_dataset2/images/train")))
print("val images:", len(os.listdir("/content/small_dataset2/images/val")))
print("test images:", len(os.listdir("/content/small_dataset2/images/test")))

train images: 500
val images: 100
test images: 100


In [116]:
import os
import random
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def show_yolo_sample(images_dir, labels_dir):
    images = os.listdir(images_dir)
    img_file = random.choice(images)

    img_path = os.path.join(images_dir, img_file)
    label_path = os.path.join(labels_dir, img_file.replace(".jpg", ".txt").replace(".png", ".txt"))

    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    h, w, _ = image.shape

    fig, ax = plt.subplots(1)
    ax.imshow(image)

    if os.path.exists(label_path):

        with open(label_path) as f:
            boxes = f.readlines()

        for box in boxes:

            cls, x, y, bw, bh = map(float, box.split())

            # convert YOLO -> pixel coords
            x_center = x * w
            y_center = y * h
            box_w = bw * w
            box_h = bh * h

            x1 = x_center - box_w / 2
            y1 = y_center - box_h / 2

            rect = patches.Rectangle(
                (x1, y1),
                box_w,
                box_h,
                linewidth=2,
                edgecolor="red",
                facecolor="none"
            )

            ax.add_patch(rect)
            ax.text(x1, y1, f"{int(cls)}", color="yellow")

    plt.axis("off")
    plt.show()


In [117]:
images_dir = "/content/small_dataset2/images/train"
labels_dir = "/content/small_dataset2/labels/train"

for _ in range(5):
    show_yolo_sample(images_dir, labels_dir)

# Model: YOLOv5m

In [76]:
!pip install ultralytics

In [181]:
import torch
from ultralytics import YOLO

yolo_model = YOLO("yolov5su.pt")

In [182]:
# Como se vera despues, no utilizamos dataloders en modelos con ultralytics
# Necesitamos solo un archivo yaml que detalle los contenido de donde estan las imagenes
yaml_content = """
path: /content/small_dataset2
train: images/train
val: images/val
test: images/test
nc: 1
names:
  0: product
"""

with open("/content/sku_dataset.yaml", "w") as f:
    f.write(yaml_content)

# Train

In [ ]:
from google.colab import drive
# Conectar con drive para guardar progreso de modelos
drive.mount('/content/drive')

In [ ]:
import os

SAVE_DIR = "/content/drive/MyDrive/sku_models"
os.makedirs(SAVE_DIR, exist_ok=True)

In [184]:
from ultralytics import YOLO

yolo_model = YOLO("yolov5su.pt")

# Congelar todo excepto el cabezal
def freeze_backbone(trainer):
    model = trainer.model
    for param in model.parameters():
        param.requires_grad = False
    for param in model.model[-1].parameters():
        param.requires_grad = True

# Para modelos con ultravision, ya viene integrada la funcion de entrenar
# Asi que solo es necesario agregar los parametros en la llamada de entro y ya
yolo_model.add_callback("on_train_start", freeze_backbone)

yolo_model.train(
    data   = "/content/sku_dataset.yaml",   # yaml generado previamente
    epochs = 10,
    imgsz  = 640,
    batch  = 4,
    lr0    = 1e-4,
    patience = 3,
    project  = "/content/drive/MyDrive/sku_models", # direccion de drive
    name     = "yolov5_1class",
)

Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/sku_dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov5su.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov5_1class, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patienc

KeyboardInterrupt: 

# Eval

In [185]:
from ultralytics import YOLO
best_model = YOLO("/content/drive/MyDrive/sku_models/yolov5_1class/weights/best.pt")

In [186]:
metrics = best_model.val(
    data    = "/content/sku_dataset.yaml",
    imgsz   = 640,
    conf    = 0.1,    # umbral bajo para priorizar recall
    iou     = 0.5,
    split   = "val",  # cambia a "test" si quieres evaluar sobre test
)

print(f"mAP@0.5:        {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95:   {metrics.box.map:.4f}")
print(f"Precisión:      {metrics.box.mp:.4f}")
print(f"Recall:         {metrics.box.mr:.4f}")

Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLOv5s summary (fused): 85 layers, 9,111,923 parameters, 0 gradients, 23.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2030.7±729.2 MB/s, size: 1126.1 KB)
val: Scanning /content/small_dataset2/labels/val.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 21.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 14.1s/it 1:39
                   all        100      15188      0.645      0.562       0.55      0.244
Speed: 10.6ms preprocess, 776.0ms inference, 0.0ms loss, 3.7ms postprocess per image
Results saved to /content/yolov5/yolov5/yolov5/runs/detect/val
mAP@0.5:        0.5499
mAP@0.5:0.95:   0.2438
Precisión:      0.6453
Recall:         0.5616


In [188]:
# ── 3. Mostrar 3 imágenes de prueba con sus detecciones ─────────────────────
import os, random, cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches

test_img_dir = "/content/small_dataset2/images/test"
test_imgs    = random.sample(os.listdir(test_img_dir), 3)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, img_name in zip(axes, test_imgs):
    img_path = os.path.join(test_img_dir, img_name)

    # Inferencia
    results = best_model(img_path, conf=0.1, iou=0.5, verbose=False)[0]

    # Imagen base
    image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (640, 640))
    ax.imshow(image)

    # Dibujar cajas
    for box in results.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        score = box.conf[0].item()

        # Escalar coords al tamaño mostrado (640)
        orig_h, orig_w = cv2.imread(img_path).shape[:2]
        x1 = x1 * 640 / orig_w
        x2 = x2 * 640 / orig_w
        y1 = y1 * 640 / orig_h
        y2 = y2 * 640 / orig_h

        ax.add_patch(patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=1.5, edgecolor="lime", facecolor="none"
        ))
        ax.text(x1, y1 - 4, f"{score:.2f}", color="lime",
                fontsize=7, backgroundcolor="black")

    n_det = len(results.boxes)
    ax.set_title(f"{img_name}\n{n_det} detecciones", fontsize=9)
    ax.axis("off")

plt.suptitle("Predicciones YOLOv5su — set de prueba (conf ≥ 0.1)", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()